# EDA -- earnings-lens synthetic corpus

Exploratory look at the 30-transcript synthetic corpus and its NLP results. **All data here is synthetic** (see README limitations).

In [1]:
import sys

sys.path.insert(0, '.')
import pandas as pd

from store.db import get_session
from store.models import FinancialOutcome, NlpResult, Transcript

pd.set_option('display.max_columns', None)
session = get_session()
nlp_df = pd.DataFrame([{c.name: getattr(r, c.name) for c in NlpResult.__table__.columns} for r in session.query(NlpResult).all()])
t_df = pd.DataFrame([{c.name: getattr(r, c.name) for c in Transcript.__table__.columns} for r in session.query(Transcript).all()])
o_df = pd.DataFrame([{c.name: getattr(r, c.name) for c in FinancialOutcome.__table__.columns} for r in session.query(FinancialOutcome).all()])
session.close()
print(len(nlp_df), 'nlp rows |', len(t_df), 'transcript rows |', len(o_df), 'outcome rows')

30 nlp rows | 30 transcript rows | 30 outcome rows


## Sentiment distribution by company

In [2]:
nlp_df.groupby('ticker')['sentiment_weighted_score'].agg(['mean', 'std', 'count'])

,mean,std,count
ticker,,,
HDFCBANK.NS,0.182667,0.283855,6
INFY.NS,0.109133,0.218314,6
RELIANCE.NS,0.156950,0.318298,6
TATAMOTORS.NS,0.187517,0.324798,6
TCS.NS,0.253967,0.240205,6


## Hedging vs certainty by company

In [3]:
nlp_df.groupby('ticker')[['hedging_score', 'certainty_score', 'avg_evasiveness']].mean().sort_values('hedging_score', ascending=False)

,hedging_score,certainty_score,avg_evasiveness
ticker,,,
TCS.NS,0.021717,0.007183,0.966667
HDFCBANK.NS,0.020617,0.008400,1.000000
RELIANCE.NS,0.020100,0.006083,0.833333
TATAMOTORS.NS,0.018900,0.007250,0.933333
INFY.NS,0.014950,0.009583,0.866667


## Guidance extraction accuracy by company

In [4]:
nlp_df.groupby('ticker')['guidance_accuracy'].mean().dropna().sort_values(ascending=False)

ticker
TATAMOTORS.NS    0.722217
INFY.NS          0.700000
HDFCBANK.NS      0.666667
RELIANCE.NS      0.600000
TCS.NS           0.200000
Name: guidance_accuracy, dtype: float64

## Readability (Flesch-Kincaid grade) distribution

In [5]:
nlp_df['flesch_kincaid_grade'].describe()

count    30.000000
mean      7.942333
std       0.849510
min       5.910000
25%       7.440000
50%       8.025000
75%       8.317500
max      10.120000
Name: flesch_kincaid_grade, dtype: float64

## Sentiment vs sector

In [6]:
merged = nlp_df.merge(t_df[['ticker', 'quarter', 'year', 'sector']], on=['ticker', 'quarter', 'year'])
merged.groupby('sector')['sentiment_weighted_score'].mean().sort_values(ascending=False)

sector
Automotive      0.187517
Banking         0.182667
IT Services     0.181550
Conglomerate    0.156950
Name: sentiment_weighted_score, dtype: float64

## Which engine ran (fallback vs real model)?

In [7]:
nlp_df[['sentiment_engine', 'topics_engine']].drop_duplicates()

,sentiment_engine,topics_engine
0,lexicon_fallback,seed_keyword_fallback
